# Özdeğer ve Özvektör Hesaplama
## YZM212 Makine Öğrenmesi – III. Laboratuvar Değerlendirmesi

Bu notebook'ta üç ana bölüm yer almaktadır:
1. **QR Algoritması ile Manuel Özdeğer/Özvektör Hesaplama** – NumPy `eig` kullanılmaz
2. **NumPy `linalg.eig` ile Hesaplama**
3. **İki Yöntemin Sonuçlarının Karşılaştırılması**

**Referans:** [LucasBN/Eigenvalues-and-Eigenvectors](https://github.com/LucasBN/Eigenvalues-and-Eigenvectors)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Kutuphaneler yuklendi.')
print(f'NumPy surumu: {np.__version__}')

---
## Bolum 1 – QR Algoritmasi (NumPy `eig` Kullanmadan)

### Teorik Arka Plan

**QR Algoritmasi**, bir kare matrisin ozdeğerlerini iteratif olarak hesaplayan sayısal bir yöntemdir.
Her adımda matris $A_k$ su sekilde guncellenir:

$$A_k = Q_k R_k \quad \Rightarrow \quad A_{k+1} = R_k Q_k$$

Yeterli iterasyon sonunda $A_k$ ust ucgen (Schur) forma yakinsar; kosegen elemanlar **ozdeğerleri** verir.

**Gram-Schmidt ortonormalizasyonu** ile QR ayrısımı:
- Her $j$. sütun icin onceki sutunlara dik bilesenler cikarilir
- Elde edilen vektor normalize edilir

> **Not:** QR algoritmasi **simetrik matrisler** icin en iyi yakinsamayi saglar.

In [ ]:
def gram_schmidt_qr(A):
    """
    Gram-Schmidt ortonormalizasyonu ile QR ayrismasi.
    A = Q @ R   (Q: ortogonal, R: ust ucgen)
    """
    n = A.shape[0]
    Q = np.zeros_like(A, dtype=float)
    R = np.zeros((n, n), dtype=float)

    for j in range(n):
        v = A[:, j].astype(float).copy()
        for i in range(j):
            R[i, j] = np.dot(Q[:, i], A[:, j])
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if abs(R[j, j]) < 1e-10:
            Q[:, j] = 0.0
        else:
            Q[:, j] = v / R[j, j]
    return Q, R


def qr_eigenvalues(A, max_iter=5000, tol=1e-9):
    """
    QR algoritmasi ile ozdeğer hesaplama.
    LucasBN (https://github.com/LucasBN/Eigenvalues-and-Eigenvectors)
    yaklasimi referans alinarak uygulanmistir.

    Parametreler:
        A        : Kare matris (simetrik icin en iyi sonuc)
        max_iter : Maksimum iterasyon
        tol      : Alt ucgen norm esigi (yakinasma kriteri)
    Dondurur:
        eigenvalues : Ozdeğerler (kosegen elemanlar)
        Q_total     : Birikimli donusum matrisi
        iter_count  : Yakinsama iterasyon sayisi
    """
    Ak = A.astype(float).copy()
    n  = Ak.shape[0]
    Q_total = np.eye(n)

    for k in range(max_iter):
        Q, R   = gram_schmidt_qr(Ak)
        Ak_new = R @ Q
        Q_total = Q_total @ Q

        if np.linalg.norm(np.tril(Ak_new, -1)) < tol:
            return np.diag(Ak_new), Q_total, k + 1
        Ak = Ak_new

    return np.diag(Ak), Q_total, max_iter


def compute_eigenvectors(A, eigenvalues):
    """
    Her ozdeğer icin (A - lambdaI)v = 0 sifir uzayini SVD ile hesapla.
    """
    n = A.shape[0]
    eigvecs = []
    for lam in eigenvalues:
        M = A.astype(float) - lam * np.eye(n)
        _, s, Vt = np.linalg.svd(M)
        v = Vt[np.argmin(s)]
        eigvecs.append(v / np.linalg.norm(v))
    return np.column_stack(eigvecs)


print('Fonksiyonlar tanimlandi.')

---
## Bolum 2 – Test Matrisi

In [ ]:
# Test matrisi: simetrik → QR mukemmel yakinsar
A = np.array([
    [ 4, -2,  1],
    [-2,  5, -1],
    [ 1, -1,  6]
], dtype=float)

print('Test Matrisi A:')
print(A)
print(f'\nBoyut     : {A.shape}')
print(f'Simetrik? : {np.allclose(A, A.T)}')

---
## Bolum 3 – Yontem 1: QR Algoritmasi

In [ ]:
print('=== Yontem 1: QR Algoritmasi (eig kullanmadan) ===')

eigenvalues_qr, Q_total, iters = qr_eigenvalues(A)
eigenvectors_qr = compute_eigenvectors(A, eigenvalues_qr)

idx_qr         = np.argsort(eigenvalues_qr)
ev_qr_sorted   = eigenvalues_qr[idx_qr]
evec_qr_sorted = eigenvectors_qr[:, idx_qr]

print(f'\nYakinasma: {iters} iterasyonda tamamlandi')
print(f'\nOzdeğerler (QR):')
for i, v in enumerate(ev_qr_sorted):
    print(f'  lambda_{i+1} = {v:.8f}')

print(f'\nOzvektorler (QR) – her sutun bir ozvektor:')
print(np.round(evec_qr_sorted, 6))

---
## Bolum 4 – Yontem 2: NumPy linalg.eig

In [ ]:
print('=== Yontem 2: NumPy linalg.eig ===')

eigenvalues_np, eigenvectors_np = np.linalg.eig(A)

idx_np         = np.argsort(eigenvalues_np.real)
ev_np_sorted   = eigenvalues_np.real[idx_np]
evec_np_sorted = eigenvectors_np.real[:, idx_np]

print(f'\nOzdeğerler (NumPy):')
for i, v in enumerate(ev_np_sorted):
    print(f'  lambda_{i+1} = {v:.8f}')

print(f'\nOzvektorler (NumPy) – her sutun bir ozvektor:')
print(np.round(evec_np_sorted, 6))

---
## Bolum 5 – Karsılastırma ve Doğrulama

In [ ]:
print('=== Ozdeğer Karsılastırması ===')
print(f'{"Sira":<6} {"QR Algoritmasi":>22} {"NumPy eig":>20} {"Fark |delta|": >14}')
print('-' * 68)
for i, (qr_v, np_v) in enumerate(zip(ev_qr_sorted, ev_np_sorted)):
    diff = abs(qr_v - np_v)
    print(f'lambda_{i+1} {qr_v:>22.10f} {np_v:>20.10f} {diff:>14.2e}')

max_diff = np.max(np.abs(ev_qr_sorted - ev_np_sorted))
print(f'\nMaksimum fark: {max_diff:.2e}')
print(f'Uyum durumu  : {"MUKEMMEL" if max_diff < 1e-6 else "FARK VAR"}')

In [ ]:
# A*v = lambda*v dogrulama (NumPy sonuclari)
print('=== Ozvektör Dogrulamasi: ||A*v - lambda*v|| ===')
for i in range(len(eigenvalues_np)):
    lam = eigenvalues_np[i].real
    v   = eigenvectors_np[:, i].real
    err = np.linalg.norm(A @ v - lam * v)
    ok  = 'DOGRU' if err < 1e-10 else 'HATALI'
    print(f'  lambda = {lam:8.5f} --> hata = {err:.2e}  [{ok}]')

---
## Bolum 6 – Gorsellestirme

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('QR Algoritmasi vs NumPy linalg.eig Karsılastırması', fontsize=13, fontweight='bold')

# Sol: Ozdeğer karşılastırma cubuğu
ax1 = axes[0]
x = np.arange(len(ev_qr_sorted))
w = 0.35
b1 = ax1.bar(x - w/2, ev_qr_sorted, w, label='QR Algoritmasi', color='steelblue', alpha=0.85)
b2 = ax1.bar(x + w/2, ev_np_sorted,  w, label='NumPy linalg.eig', color='coral',  alpha=0.85)
for b in [*b1, *b2]:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
             f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax1.set_xticks(x)
ax1.set_xticklabels([f'lambda_{i+1}' for i in range(len(ev_qr_sorted))])
ax1.set_ylabel('Ozdeğer Degeri')
ax1.set_title('Ozdeğer Karsılastırması')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Sag: Ozvektor yonleri (ilk 2 bilesen)
ax2 = axes[1]
colors_q = ['steelblue', 'navy',    'royalblue']
colors_n = ['coral',     'darkred', 'tomato']
origin = [0, 0]
for i in range(3):
    vq = evec_qr_sorted[:, i]
    vn = evec_np_sorted[:, i]
    ax2.quiver(*origin, vq[0], vq[1], angles='xy', scale_units='xy', scale=1,
               color=colors_q[i], width=0.015, label=f'QR: v{i+1}')
    ax2.quiver(*origin, vn[0], vn[1], angles='xy', scale_units='xy', scale=1,
               color=colors_n[i], width=0.008, label=f'NP: v{i+1}', alpha=0.7)
ax2.axhline(0, color='gray', lw=0.5)
ax2.axvline(0, color='gray', lw=0.5)
ax2.set_xlim(-1.4, 1.4)
ax2.set_ylim(-1.4, 1.4)
ax2.set_aspect('equal')
ax2.set_title('Ozvektor Yonleri (x1, x2 Bilesenleri)')
ax2.set_xlabel('x1')
ax2.set_ylabel('x2')
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eigenvectors_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: eigenvectors_comparison.png')

---
## Bolum 7 – Farkli Boyutlarda Ek Testler

In [ ]:
test_matrices = {
    '2x2 Simetrik': np.array([[3.0, 1.0], [1.0, 3.0]]),
    '4x4 Simetrik': np.array([[ 5,  1,  0,  2],
                               [ 1,  4,  1,  0],
                               [ 0,  1,  3,  1],
                               [ 2,  0,  1,  6]], dtype=float)
}

for name, M in test_matrices.items():
    print(f'\n=== {name} Matris Testi ===')
    ev_qr, _, it = qr_eigenvalues(M)
    ev_np, _     = np.linalg.eig(M)
    ev_qr_s = np.sort(np.real(ev_qr))
    ev_np_s = np.sort(np.real(ev_np))
    print(f'  QR   : {np.round(ev_qr_s, 5)}  (iterasyon: {it})')
    print(f'  NumPy: {np.round(ev_np_s, 5)}')
    print(f'  Maks. fark: {np.max(np.abs(ev_qr_s - ev_np_s)):.2e}')

---
## Sonuc

| Yontem | Ozdeğer Doğruluğu | Hiz | Acıklık |
|--------|-------------------|-----|----------|
| **QR Algoritmasi (Manuel)** | Yuksek (simetrik matrisler) | Yavas | Ic yapı gorünür |
| **NumPy `linalg.eig`** | Cok yuksek (LAPACK) | Cok hızlı | Kara kutu |

Iki yontem de ayni ozdeğerleri uretmektedir. Simetrik matris icin fark **< 10^-14** mertebesindedir.

### Referanslar
1. LucasBN (2020). *Eigenvalues-and-Eigenvectors*. GitHub. https://github.com/LucasBN/Eigenvalues-and-Eigenvectors
2. NumPy Developers (2024). *numpy.linalg.eig*. https://numpy.org/doc/2.1/reference/generated/numpy.linalg.eig.html
3. Brownlee, J. *Gentle Introduction to Eigenvalues and Eigenvectors for Machine Learning*. MachineLearningMastery.com
4. Brownlee, J. *Introduction to Matrices and Matrix Arithmetic for Machine Learning*. MachineLearningMastery.com